<div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 30px; border-radius: 10px; color: white;'>
    <h1 style='margin: 0; font-size: 36px; font-weight: bold;'>🎯 Clustering in Practice</h1>
    <p style='margin: 10px 0 0 0; font-size: 18px; opacity: 0.9;'>Hands-On Session: K-Means & DBSCAN for Real-World Segmentation</p>
</div>

---

## 📋 Session Overview

In this hands-on notebook, you'll apply **K-Means** and **DBSCAN** clustering algorithms to real customer segmentation data. You'll learn to:

- ✅ Apply K-Means with k-means++ initialization
- ✅ Determine optimal clusters using elbow method and silhouette scores
- ✅ Use DBSCAN for density-based clustering with noise handling
- ✅ Visualize high-dimensional clusters using PCA
- ✅ Compare algorithmic approaches for different data characteristics

**Estimated time:** 90-110 minutes

<div style='background-color: #e7f3ff; border-left: 5px solid #2196F3; padding: 15px; margin: 20px 0;'>
    <h3 style='margin-top: 0; color: #1976D2;'>🎯 Learning Goals</h3>
    <ol style='margin-bottom: 0;'>
        <li>Implement K-Means clustering with optimal parameter selection</li>
        <li>Apply DBSCAN for irregular cluster shapes and outlier detection</li>
        <li>Validate cluster quality using silhouette scores</li>
        <li>Visualize clusters in 2D using PCA dimensionality reduction</li>
        <li>Make informed algorithmic choices based on data characteristics</li>
    </ol>
</div>

<div style='background-color: #4CAF50; padding: 15px; border-radius: 5px; margin: 30px 0 20px 0;'>
    <h2 style='margin: 0; color: white; font-weight: bold;'>MODULE 1: Setup & Data Loading</h2>
</div>

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.datasets import make_blobs, make_moons
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print('✅ Libraries imported successfully')

### 📊 Dataset Introduction

We'll work with a **customer segmentation dataset** containing:
- **Annual Income** (in thousands)
- **Spending Score** (1-100 scale based on purchasing behavior)

**Business objective:** Identify distinct customer segments to tailor marketing strategies.

In [ ]:
# Generate synthetic customer data with distinct segments
np.random.seed(42)

# Create 5 natural customer segments
# Low income, low spending
segment1 = np.random.randn(40, 2) * [5, 10] + [30, 30]

# Low income, high spending
segment2 = np.random.randn(40, 2) * [5, 10] + [30, 75]

# Medium income, medium spending
segment3 = np.random.randn(50, 2) * [8, 12] + [60, 50]

# High income, low spending
segment4 = np.random.randn(40, 2) * [6, 10] + [85, 30]

# High income, high spending
segment5 = np.random.randn(40, 2) * [6, 10] + [85, 75]

# Add some noise points (outliers)
noise = np.random.uniform(low=[20, 10], high=[100, 90], size=(10, 2))

# Combine all segments
X = np.vstack([segment1, segment2, segment3, segment4, segment5, noise])

# Create DataFrame
df = pd.DataFrame(X, columns=['Annual_Income_k$', 'Spending_Score'])

print(f'✅ Generated {len(df)} customer records')
print(f'Features: {list(df.columns)}')
df.head()

<div style='background-color: #FF9800; padding: 15px; border-radius: 5px; margin: 30px 0 20px 0;'>
    <h2 style='margin: 0; color: white; font-weight: bold;'>MODULE 2: Exploratory Data Analysis</h2>
</div>

In [ ]:
# Display summary statistics
print('📊 Dataset Summary Statistics:')
print(df.describe())

print('\n📋 Data Info:')
print(df.info())

print('\n🔍 Missing Values:')
print(df.isnull().sum())

In [ ]:
# Visualize the raw data distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Scatter plot
axes[0].scatter(df['Annual_Income_k$'], df['Spending_Score'],
                alpha=0.6, s=80, edgecolors='black', linewidth=0.5)
axes[0].set_xlabel('Annual Income (k$)', fontsize=12)
axes[0].set_ylabel('Spending Score', fontsize=12)
axes[0].set_title('Customer Distribution', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Income distribution
axes[1].hist(df['Annual_Income_k$'], bins=20, edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Annual Income (k$)', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Income Distribution', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

# Spending score distribution
axes[2].hist(df['Spending_Score'], bins=20, edgecolor='black', alpha=0.7, color='coral')
axes[2].set_xlabel('Spending Score', fontsize=12)
axes[2].set_ylabel('Frequency', fontsize=12)
axes[2].set_title('Spending Score Distribution', fontsize=14, fontweight='bold')
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print('\n💡 Observation: The scatter plot suggests multiple distinct customer groups!')

<div style='background-color: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; margin: 20px 0;'>
    <h4 style='margin-top: 0; color: #856404;'>💡 Key Insight</h4>
    <p style='margin-bottom: 0;'>The scatter plot reveals potential clusters at the corners and center, suggesting natural customer segments based on income-spending behavior patterns.</p>
</div>

<div style='background-color: #9C27B0; padding: 15px; border-radius: 5px; margin: 30px 0 20px 0;'>
    <h2 style='margin: 0; color: white; font-weight: bold;'>MODULE 3: Data Preprocessing</h2>
</div>

In [ ]:
# Standardize features (critical for distance-based algorithms)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

print('✅ Features standardized (zero mean, unit variance)')
print(f'Original data shape: {df.shape}')
print(f'Scaled data shape: {X_scaled.shape}')
print(f'\nScaled data mean: {X_scaled.mean(axis=0)}')
print(f'Scaled data std: {X_scaled.std(axis=0)}')

<div style='background-color: #ffebee; border-left: 5px solid #f44336; padding: 15px; margin: 20px 0;'>
    <h4 style='margin-top: 0; color: #c62828;'>⚠️ Critical Step</h4>
    <p style='margin-bottom: 0;'><strong>Why standardization matters:</strong> K-Means and DBSCAN use Euclidean distance. Without scaling, features with larger ranges (like Income: 20-100) dominate over smaller ranges (Spending: 1-100), distorting cluster formation.</p>
</div>

<div style='background-color: #2196F3; padding: 15px; border-radius: 5px; margin: 30px 0 20px 0;'>
    <h2 style='margin: 0; color: white; font-weight: bold;'>MODULE 4: K-Means Clustering</h2>
</div>

### 🔍 Step 1: Determine Optimal K using Elbow Method

In [ ]:
# Calculate WCSS for different K values
wcss = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

# Plot elbow curve
plt.figure(figsize=(10, 6))
plt.plot(K_range, wcss, marker='o', linewidth=2, markersize=10)
plt.xlabel('Number of Clusters (K)', fontsize=12)
plt.ylabel('WCSS (Inertia)', fontsize=12)
plt.title('Elbow Method for Optimal K', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.xticks(K_range)
plt.tight_layout()
plt.show()

print('💡 Look for the "elbow" where WCSS decrease slows significantly')

### 📊 Step 2: Validate with Silhouette Scores

In [ ]:
# Calculate silhouette scores for different K
silhouette_scores = []

for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)
    print(f'K={k}: Silhouette Score = {score:.3f}')

# Plot silhouette scores
plt.figure(figsize=(10, 6))
plt.plot(range(2, 11), silhouette_scores, marker='s', linewidth=2,
         markersize=10, color='green')
plt.xlabel('Number of Clusters (K)', fontsize=12)
plt.ylabel('Silhouette Score', fontsize=12)
plt.title('Silhouette Analysis', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.xticks(range(2, 11))
plt.axhline(y=0.5, color='red', linestyle='--', label='Good threshold (0.5)')
plt.legend()
plt.tight_layout()
plt.show()

optimal_k = silhouette_scores.index(max(silhouette_scores)) + 2
print(f'\n✅ Optimal K based on silhouette score: {optimal_k}')

### 🎯 Step 3: Apply K-Means with Optimal K

In [ ]:
# Fit K-Means with optimal K
optimal_k = 5  # Based on analysis above
kmeans = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42)
kmeans_labels = kmeans.fit_predict(X_scaled)

# Add cluster labels to dataframe
df['KMeans_Cluster'] = kmeans_labels

print(f'✅ K-Means clustering completed with K={optimal_k}')
print(f'Cluster distribution:')
print(df['KMeans_Cluster'].value_counts().sort_index())

# Calculate final silhouette score
final_silhouette = silhouette_score(X_scaled, kmeans_labels)
print(f'\nFinal Silhouette Score: {final_silhouette:.3f}')

In [ ]:
# Visualize K-Means clusters
plt.figure(figsize=(10, 7))
scatter = plt.scatter(df['Annual_Income_k$'], df['Spending_Score'],
                      c=kmeans_labels, cmap='viridis',
                      s=100, alpha=0.6, edgecolors='black', linewidth=0.5)

# Plot centroids
centroids = scaler.inverse_transform(kmeans.cluster_centers_)
plt.scatter(centroids[:, 0], centroids[:, 1],
            c='red', s=300, marker='X', edgecolors='black', linewidth=2,
            label='Centroids')

plt.xlabel('Annual Income (k$)', fontsize=12)
plt.ylabel('Spending Score', fontsize=12)
plt.title('K-Means Clustering Results', fontsize=14, fontweight='bold')
plt.colorbar(scatter, label='Cluster')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

<div style='background-color: #E91E63; padding: 15px; border-radius: 5px; margin: 30px 0 20px 0;'>
    <h2 style='margin: 0; color: white; font-weight: bold;'>MODULE 5: DBSCAN Clustering</h2>
</div>

### 🔍 Step 1: Determine eps using K-Distance Graph

In [ ]:
from sklearn.neighbors import NearestNeighbors

# Use min_samples = 2 * dimensionality = 2 * 2 = 4
min_samples = 4

# Compute k-nearest neighbors
neighbors = NearestNeighbors(n_neighbors=min_samples)
neighbors.fit(X_scaled)
distances, indices = neighbors.kneighbors(X_scaled)

# Sort distances to 4th nearest neighbor
distances = np.sort(distances[:, -1], axis=0)

# Plot k-distance graph
plt.figure(figsize=(10, 6))
plt.plot(distances, linewidth=2)
plt.xlabel('Data Points (sorted)', fontsize=12)
plt.ylabel(f'{min_samples}-NN Distance', fontsize=12)
plt.title('K-Distance Graph for eps Selection', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.axhline(y=0.5, color='red', linestyle='--', label='Suggested eps=0.5')
plt.legend()
plt.tight_layout()
plt.show()

print('💡 The elbow point in this curve suggests the optimal eps value')
print('💡 eps should be slightly above where the curve starts rising sharply')

### 🎯 Step 2: Apply DBSCAN with Tuned Parameters

In [ ]:
# Apply DBSCAN
eps = 0.5
min_samples = 5

dbscan = DBSCAN(eps=eps, min_samples=min_samples)
dbscan_labels = dbscan.fit_predict(X_scaled)

# Add labels to dataframe
df['DBSCAN_Cluster'] = dbscan_labels

# Count clusters and noise
n_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise = list(dbscan_labels).count(-1)

print(f'✅ DBSCAN clustering completed')
print(f'eps={eps}, min_samples={min_samples}')
print(f'\nNumber of clusters: {n_clusters}')
print(f'Number of noise points: {n_noise} ({n_noise/len(df)*100:.1f}%)')
print(f'\nCluster distribution:')
print(df['DBSCAN_Cluster'].value_counts().sort_index())

# Silhouette score (excluding noise points)
if n_clusters > 1:
    mask = dbscan_labels != -1
    if sum(mask) > 0:
        dbscan_silhouette = silhouette_score(X_scaled[mask], dbscan_labels[mask])
        print(f'\nSilhouette Score (excluding noise): {dbscan_silhouette:.3f}')

In [ ]:
# Visualize DBSCAN clusters
plt.figure(figsize=(10, 7))

# Separate noise points
core_samples_mask = np.zeros_like(dbscan_labels, dtype=bool)
core_samples_mask[dbscan.core_sample_indices_] = True

unique_labels = set(dbscan_labels)
colors = plt.cm.Spectral(np.linspace(0, 1, len(unique_labels)))

for k, col in zip(unique_labels, colors):
    if k == -1:
        # Noise points in black
        col = 'black'
        marker = 'x'
        label = 'Noise'
    else:
        marker = 'o'
        label = f'Cluster {k}'

    class_member_mask = (dbscan_labels == k)
    xy = df.loc[class_member_mask, ['Annual_Income_k$', 'Spending_Score']].values

    plt.scatter(xy[:, 0], xy[:, 1], c=[col], marker=marker,
                s=100, alpha=0.6, edgecolors='black', linewidth=0.5,
                label=label)

plt.xlabel('Annual Income (k$)', fontsize=12)
plt.ylabel('Spending Score', fontsize=12)
plt.title('DBSCAN Clustering Results', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\n💡 Note: Black X markers represent noise points (outliers)')

<div style='background-color: #00BCD4; padding: 15px; border-radius: 5px; margin: 30px 0 20px 0;'>
    <h2 style='margin: 0; color: white; font-weight: bold;'>MODULE 6: Algorithm Comparison</h2>
</div>

In [ ]:
# Side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# K-Means
scatter1 = axes[0].scatter(df['Annual_Income_k$'], df['Spending_Score'],
                           c=df['KMeans_Cluster'], cmap='viridis',
                           s=100, alpha=0.6, edgecolors='black', linewidth=0.5)
axes[0].scatter(centroids[:, 0], centroids[:, 1],
                c='red', s=300, marker='X', edgecolors='black', linewidth=2)
axes[0].set_xlabel('Annual Income (k$)', fontsize=12)
axes[0].set_ylabel('Spending Score', fontsize=12)
axes[0].set_title(f'K-Means (K={optimal_k})', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# DBSCAN
for k in set(dbscan_labels):
    if k == -1:
        col = 'black'
        marker = 'x'
    else:
        col = plt.cm.Spectral(k / max(set(dbscan_labels)))
        marker = 'o'

    mask = (dbscan_labels == k)
    axes[1].scatter(df.loc[mask, 'Annual_Income_k$'],
                    df.loc[mask, 'Spending_Score'],
                    c=[col], marker=marker, s=100, alpha=0.6,
                    edgecolors='black', linewidth=0.5)

axes[1].set_xlabel('Annual Income (k$)', fontsize=12)
axes[1].set_ylabel('Spending Score', fontsize=12)
axes[1].set_title(f'DBSCAN (eps={eps}, min_samples={min_samples})',
                  fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 📊 Quantitative Comparison

| Metric | K-Means | DBSCAN |
|--------|---------|--------|
| Number of clusters | 5 (predefined) | Auto-detected |
| Silhouette score | Calculated for all | Calculated excluding noise |
| Handles outliers | No (forces into clusters) | Yes (labels as noise) |
| Cluster shape | Assumes spherical | Arbitrary shapes |
| Requires K | Yes | No |

<div style='background-color: #673AB7; padding: 15px; border-radius: 5px; margin: 30px 0 20px 0;'>
    <h2 style='margin: 0; color: white; font-weight: bold;'>MODULE 7: PCA Visualization</h2>
</div>

For higher-dimensional datasets, **PCA (Principal Component Analysis)** helps visualize clusters by reducing dimensions to 2D while preserving variance.

In [ ]:
# Apply PCA for 2D visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f'✅ PCA applied: {X_scaled.shape} → {X_pca.shape}')
print(f'Explained variance ratio: {pca.explained_variance_ratio_}')
print(f'Total variance preserved: {sum(pca.explained_variance_ratio_)*100:.1f}%')

In [ ]:
# Visualize in PCA space
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# K-Means in PCA space
axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=kmeans_labels,
                cmap='viridis', s=100, alpha=0.6,
                edgecolors='black', linewidth=0.5)
axes[0].set_xlabel('PC1', fontsize=12)
axes[0].set_ylabel('PC2', fontsize=12)
axes[0].set_title('K-Means in PCA Space', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)x

# DBSCAN in PCA space
for k in set(dbscan_labels):
    if k == -1:
        col = 'black'
        marker = 'x'
    else:
        col = plt.cm.Spectral(k / max(set(dbscan_labels)))
        marker = 'o'

    mask = (dbscan_labels == k)
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=[col], marker=marker, s=100, alpha=0.6,
                    edgecolors='black', linewidth=0.5)

axes[1].set_xlabel('PC1', fontsize=12)
axes[1].set_ylabel('PC2', fontsize=12)
axes[1].set_title('DBSCAN in PCA Space', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

<div style='background-color: #FF5722; padding: 15px; border-radius: 5px; margin: 30px 0 20px 0;'>
    <h2 style='margin: 0; color: white; font-weight: bold;'>MODULE 8: Experimentation</h2>
</div>

### 🧪 Try These Experiments:

1. **Test different K values** in K-Means (K=3, K=7) and observe silhouette score changes
2. **Vary DBSCAN parameters:**
   - Try eps=0.3, eps=0.7
   - Try min_samples=3, min_samples=10
3. **Generate non-spherical clusters** using `make_moons` dataset and compare K-Means vs DBSCAN
4. **Add more noise points** and observe DBSCAN's robustness

In [ ]:
# Experiment: Non-spherical clusters (moon shapes)
from sklearn.datasets import make_moons

X_moons, _ = make_moons(n_samples=300, noise=0.05, random_state=42)
X_moons_scaled = StandardScaler().fit_transform(X_moons)

# K-Means (will struggle with crescent shapes)
kmeans_moons = KMeans(n_clusters=2, random_state=42)
kmeans_moons_labels = kmeans_moons.fit_predict(X_moons_scaled)

# DBSCAN (should handle well)
dbscan_moons = DBSCAN(eps=0.3, min_samples=5)
dbscan_moons_labels = dbscan_moons.fit_predict(X_moons_scaled)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=kmeans_moons_labels,
                cmap='viridis', s=50, edgecolors='black', linewidth=0.3)
axes[0].set_title('K-Means on Crescent Data', fontsize=14, fontweight='bold')

axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=dbscan_moons_labels,
                cmap='Spectral', s=50, edgecolors='black', linewidth=0.3)
axes[1].set_title('DBSCAN on Crescent Data', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print('💡 Notice: DBSCAN correctly identifies the crescent shapes!')
print('💡 K-Means struggles because it assumes spherical clusters')

<div style='background-color: #607D8B; padding: 15px; border-radius: 5px; margin: 30px 0 20px 0;'>
    <h2 style='margin: 0; color: white; font-weight: bold;'>MODULE 9: Summary & Key Takeaways</h2>
</div>

<div style='background-color: #e8f5e9; border-left: 5px solid #4CAF50; padding: 20px; margin: 20px 0;'>
    <h3 style='margin-top: 0; color: #2E7D32;'>✅ What You've Learned</h3>
    
**K-Means Clustering:**
- Applied k-means++ initialization for better convergence
- Used elbow method and silhouette scores to determine optimal K
- Understood that K-Means works best for spherical, similar-sized clusters
    
**DBSCAN Clustering:**
- Tuned eps using k-distance graphs
- Handled outliers by identifying noise points (label -1)
- Discovered clusters of arbitrary shapes without specifying K
    
**Best Practices:**
- Always standardize features before clustering
- Validate clusters using silhouette scores
- Use PCA for visualizing high-dimensional clusters
- Choose algorithm based on data characteristics (spherical vs irregular, known K vs unknown)
</div>

### 🎯 Decision Framework: Which Algorithm to Use?

**Use K-Means when:**
- ✅ You have a rough idea of the number of clusters
- ✅ Clusters are roughly spherical and similar in size
- ✅ You need fast computation for large datasets
- ✅ Data has minimal outliers

**Use DBSCAN when:**
- ✅ You don't know the number of clusters in advance
- ✅ Clusters have irregular, non-spherical shapes
- ✅ You need to identify and ignore outliers
- ✅ Clusters vary significantly in density or size

**Real-world tip:** Try both algorithms and compare results!

### 🚀 Next Steps

1. Apply clustering to your own datasets
2. Explore hierarchical clustering for dendrogram-based segmentation
3. Learn about Gaussian Mixture Models (GMM) for probabilistic clustering
4. Practice parameter tuning on different data distributions
5. Combine clustering with domain knowledge for business insights

---

<div style='text-align: center; padding: 20px; background-color: #f5f5f5; border-radius: 10px;'>
    <h3 style='color: #333;'>🎓 Session Complete!</h3>
    <p style='color: #666;'>You've successfully implemented and compared K-Means and DBSCAN clustering algorithms.</p>
</div>